# Preparing the environment

In [ ]:
library(karyoploteR)
library(GenomicRanges)
library(BSgenome.Hsapiens.UCSC.hg38)
library(ggplot2)
library(dplyr)
library(circlize)
library(tidyverse)
library(scales)

Loading required package: regioneR

Loading required package: GenomicRanges

Loading required package: stats4



Loading required package: BiocGenerics


Attaching package: ‘BiocGenerics’


The following objects are masked from ‘package:lubridate’:

    intersect, setdiff, union


The following objects are masked from ‘package:dplyr’:

    combine, intersect, setdiff, union


The following objects are masked from ‘package:stats’:

    IQR, mad, sd, var, xtabs


The following objects are masked from ‘package:base’:

    anyDuplicated, aperm, append, as.data.frame, basename, cbind,
    colnames, dirname, do.call, duplicated, eval, evalq, Filter, Find,
    get, grep, grepl, intersect, is.unsorted, lapply, Map, mapply,
    match, mget, order, paste, pmax, pmax.int, pmin, pmin.int,
    Position, rank, rbind, Reduce, rownames, sapply, setdiff, sort,
    table, tapply, union, unique, unsplit, which.max, which.min


Loading required package: S4Vectors


Attaching package: ‘S4Vectors’


The following objects are masked from ‘package:lubridate’:

    second, second<-


The following objects are masked from

# Dataviz distribution of STRs along coding and non-coding regions

In [ ]:
# =========================================================
# File Paths Configuration
# =========================================================
tsv_path <- "../../../samples/code_regions_statistics.tsv"
csv_path <- "../../../samples/others_regions_statistics.csv"
png_output <- "results/faceted_bars_regions.png"

# Create the 'results' directory if it does not exist
dir.create(dirname(png_output), showWarnings = FALSE, recursive = TRUE)

# =========================================================
# 1. Read and Prepare Data
# =========================================================
df_regions <- read_tsv(tsv_path, show_col_types = FALSE)
df_others <- read_csv(csv_path, show_col_types = FALSE)

# Ensure all column names are lowercase for consistency
colnames(df_regions) <- tolower(colnames(df_regions))
colnames(df_others) <- tolower(colnames(df_others))

# --- Coding Regions ---
df_coding <- df_regions %>%
  filter(category == "distribution" & metric != "others") %>%
  rename(name = metric) %>%
  select(name, value) %>%
  mutate(
    value = as.numeric(value), 
    group = "Coding Regions"
  )

# --- Non-Coding Regions (Others) ---
df_noncoding <- df_others %>%
  count(gene_biotype) %>%
  rename(name = gene_biotype, value = n) %>% 
  select(name, value) %>%
  mutate(
    value = as.numeric(value), 
    group = "Non-Coding Regions (Others)"
  )

# Merge datasets
df_total <- bind_rows(df_coding, df_noncoding)

# =========================================================
# 2. Map to Clean, User-Friendly Labels (COM AGRUPAMENTOS)
# =========================================================
df_total <- df_total %>%
  mutate(
    name_lower = tolower(name),
    name_grouped = case_when(
      # --- Regions from TSV  ---
      name_lower == "intron"           ~ "Intron",
      name_lower == "intergenic"       ~ "Intergenic",
      name_lower == "promoter"         ~ "Promoter",
      name_lower == "three_prime_utr"  ~ "3' UTR",
      name_lower == "non_coding_exons" ~ "Non-coding Exons",
      name_lower == "five_prime_utr"   ~ "5' UTR",
      name_lower == "cds"              ~ "CDS",
      
      # --- Biotypes from CSV  ---
      name_lower == "lncrna" ~ "lncRNA",
      
      # Group ALL pseudogenes using str_detect
      stringr::str_detect(name_lower, "pseudogene") ~ "Pseudogenes",
      
      # Groups smaller RNAs
      name_lower %in% c("snrna", "misc_rna") ~ "Other RNAs",
      
      # Group the rest (Unknown, TEC, IG V)
      name_lower %in% c("unknown", "tec", "ig_v_gene") ~ "Unknown / Misc",
      
      TRUE ~ "Others"
    )
  ) %>%
  # Since we've grouped several categories together, we need to add up their values!
  group_by(group, name = name_grouped) %>%
  summarise(value = sum(value), .groups = "drop")

# =========================================================
# 3. Build the Faceted Bar Chart (Clean Aesthetic)
# =========================================================
p_bars <- ggplot(df_total, aes(x = reorder(name, value), y = value, fill = group)) +
  
  # Slightly thinner bars (width = 0.6) and soft transparency for a modern look
  geom_col(width = 0.65, show.legend = FALSE, alpha = 0.9) +
  
  # Add exact numbers at the end of each bar
  geom_text(aes(label = scales::comma(value)), 
            hjust = -0.15, size = 3.5, fontface = "bold", color = "#555555") +
  
  # Split the panel into two (Coding vs Non-Coding) with independent axes
  facet_wrap(~group, scales = "free", ncol = 2) +
  
  # Flip axes for horizontal bars (easier to read long names)
  coord_flip(clip = "off") + 
  
  # Format numeric axis in thousands (K) and add 20% margin to the right
  scale_y_continuous(
    labels = scales::label_number(scale = 1e-3, suffix = "K"),
    expand = expansion(mult = c(0, 0.20)) 
  ) +
  
  # Custom, slightly softer color palette
  scale_fill_manual(values = c(
    "Coding Regions" = "#31688E", 
    "Non-Coding Regions (Others)" = "#35B779"
  )) +
  
  # Minimalist Theme Configuration
  theme_minimal(base_size = 12) +
  theme(
    # Strip (Facet) text styling - removed background boxes for clean look
    strip.text = element_text(size = 14, face = "bold", color = "#222222", margin = margin(b = 15)),
    strip.background = element_blank(), 
    
    # Axis styling
    axis.title = element_blank(),
    axis.text.y = element_text(size = 11, face = "bold", color = "#444444"),
    axis.text.x = element_text(size = 10, color = "#777777"),
    
    # Grid lines: keep only subtle vertical dotted lines
    panel.grid.major.y = element_blank(), 
    panel.grid.minor = element_blank(),
    panel.grid.major.x = element_line(color = "grey85", linetype = "dotted", size = 0.4),
    
    # Spacing and margins
    panel.spacing = unit(3, "lines"), 
    plot.title = element_text(size = 16, face = "bold", hjust = 0.5, margin = margin(b = 30), color = "#111111"),
    plot.margin = margin(30, 40, 30, 30),
    plot.background = element_rect(fill = "white", color = NA)
  ) +
  labs(title = "Variant Distribution Count Across Genomic Regions")

# =========================================================
# 4. Save Plot in High Resolution
# =========================================================
ggsave(png_output, plot = p_bars, width = 14, height = 8, dpi = 300, bg = "white")

cat("\n=> Success! Clean chart saved to:", png_output, "\n")


=> Success! Clean chart saved to: results/faceted_bars_regions.png 


# Dataviz STRs along the chromossomes

Data import

In [29]:
# File path
original_file <- "../../../samples/STRs_analysis_dataset.tsv"

# Read cleaned file with read.delim
df_annot <- read.delim(original_file, header=TRUE, sep="\t", stringsAsFactors = FALSE)

# Show the first lines of the updated data frame
head(df_annot)

,STRs_ID,group,age,sex,allele1_est,allele2_est,depth,repeat_unit,gene_id,gene_name,⋯,end,sample_id,pop,contribution,type,n_outliers,outlier_samples,outlier_residuals,n_clusters,noise_ratio
,<chr>,<chr>,<int>,<chr>,<dbl>,<dbl>,<int>,<chr>,<chr>,<chr>,⋯,<int>,<chr>,<chr>,<chr>,<chr>,<int>,<chr>,<chr>,<int>,<dbl>
1,chr16:12744297:AAGC:12,case,45,M,-0.5,5.35,67,AAGC,ENSG00000103381,CPPED1,⋯,12744345,REC-54-1.bam,AMR,,INSIDE,NA,,,NA,NA
2,chr3:78233947:AGTAT:0,case,45,M,0.0,5.36,84,AGTAT,.,.,⋯,78233948,REC-54-1.bam,AMR,,INSIDE,0,,,1,0
3,chr7:34047815:CCTT:0,case,45,M,0.0,6.15,69,CCTT,ENSG00000164619,BMPER,⋯,34047816,REC-54-1.bam,AMR,,INSIDE,0,,,1,0
4,chr2:182912842:AAGG:11,case,45,M,5.0,5.70,78,AAGG,ENSG00000061676,NCKAP1,⋯,182912887,REC-54-1.bam,AMR,,INSIDE,0,,,1,0
5,chr4:142544119:GT:20,case,45,M,5.5,41.41,73,GT,ENSG00000109452,INPP4B,⋯,142544158,REC-54-1.bam,AMR,,INSIDE,0,,,1,0
6,chr12:132156455:AC:186,case,38,M,0.0,154.57,48,AC,.,.,⋯,132156826,REC-97-1.bam,EUR,EUR(28.85%)|EAS(12.6%)|AMR(21.8%)|SAS(16.21%)|AFR(20.54%),CLOSEST,0,,,1,0


Number of STRs to be visualized and available chromosomes

In [30]:
# 1. Add "chr" prefix directly to the chrom column
df_annot <- df_annot %>%
  mutate(chrom = paste0("chr", chrom))

# 2. Define chromosomes with "chr" prefix
chromosomes_to_plot <- c(paste0("chr", 1:22), "chrX", "chrY")

# 3. Filter
df_filtered <- df_annot %>%
  filter(chrom %in% chromosomes_to_plot)

# 4. Create GRanges (already with correct chromosome names)
gr_strs <- GRanges(
  seqnames = df_filtered$chrom,
  ranges = IRanges(start = df_filtered$start, end = df_filtered$end)
)

# 5. Add metadata
mcols(gr_strs)$ID <- df_filtered$STRs_ID
mcols(gr_strs)$Region <- df_filtered$region
mcols(gr_strs)$Gene_Name <- df_filtered$gene_name

# 6. Verify
cat("STRs to plot:", length(gr_strs), "\n")
print(table(gr_strs$Region))
cat("\nChromosomes available for plotting:\n")
print(unique(seqnames(gr_strs)))


STRs to plot: 491178 

             CDS   five_prime_utr       intergenic           intron 
             158              902           196579           199035 
non_coding_exons           others         promoter  three_prime_utr 
            1356            81402             7628             4118 

Chromosomes available for plotting:
 [1] chr16 chr3  chr7  chr2  chr4  chr12 chr15 chr14 chr9  chr5  chrY  chr19
[13] chr17 chr20 chrX  chr10 chr22 chr21 chr8  chr11 chr13 chr1  chr6  chr18
24 Levels: chr16 chr3 chr7 chr2 chr4 chr12 chr15 chr14 chr9 chr5 chrY ... chr18


# Plots

Plot simples

Heatmap showing STR density per 1 MB

In [31]:
# Define standard somatic and sex chromosomes
chromosomes_to_plot <- c(paste0("chr", 1:22), "chrX", "chrY")

# Bin parameters
bin_size <- 1e6
chr_lengths <- seqlengths(BSgenome.Hsapiens.UCSC.hg38)

# Create bins for chromosomes of interest
bins_list <- list()
for (chr in chromosomes_to_plot) {
  if (chr %in% names(chr_lengths)) {
    chr_len <- chr_lengths[chr]
    bins_df <- data.frame(
      chr = chr,
      start = seq(1, chr_len, by = bin_size),
      end = pmin(seq(1, chr_len, by = bin_size) + bin_size - 1, chr_len)
    )
    bins_list[[chr]] <- bins_df
  }
}

bins <- do.call(rbind, bins_list)
bins_gr <- GRanges(seqnames = bins$chr,
                   ranges = IRanges(start = bins$start, end = bins$end))

# Calculate the density of STRs in bins using your complete GRanges
overlaps <- countOverlaps(bins_gr, gr_strs)
mcols(bins_gr)$density <- overlaps

# Set color palette for heatmap
colors <- c("white", "yellow","#d94701", "#ee070bff")

# Set intervals for quantitative legend
breaks <- quantile(mcols(bins_gr)$density, probs = seq(0, 1, length.out = length(colors) + 1))
legend_labels <- paste0(round(breaks[-length(breaks)]), " - ", round(breaks[-1]))

png("results/density_STRs.png", 
   width = 24, height = 14, units = "in", res = 600)


# Create karyotype plot
kp <- plotKaryotype(genome = "hg38",
                    chromosomes = chromosomes_to_plot,
                    plot.type = 1,
                    cex = 1.2)

# Plot density heatmap
kpHeatmap(kp, data = bins_gr, y = bins_gr$density, colors = colors)

# Add quantitative legend to chart
legend(x = "bottomright",
       legend = legend_labels,
       fill = colors,
       title = "STRs density (per 1 MB)",
       bg = "white",
       cex = 1.5)

# Close the graphics device to save the file
dev.off()

cat("Plot saved to: results/density_STRs.png\n")

agg_record_1002571446 
                    2

Plot saved to: results/density_STRs.png
